In [ ]:
!pip install torch numpy scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}")

## Cell 1: Config

All tunable training parameters in one place.
Data path points to Kaggle Dataset (no Drive, no MNE, no EDF).

In [ ]:
# --- PATHS ---
DATA_DIR = "/kaggle/input/datasets/rebelxhearts/npy-traintestdata/processed"

# --- TRAINING ---
BATCH_SIZE      = 8
LEARNING_RATE   = 0.001
NUM_EPOCHS      = 10
RANDOM_SEED     = 42
SEIZURE_WEIGHT  = 8.0     # missed seizures cost 8x more -> high recall

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Data: {DATA_DIR}")
print(f"Batch: {BATCH_SIZE} | LR: {LEARNING_RATE} | Epochs: {NUM_EPOCHS}")
print(f"Seizure weight: {SEIZURE_WEIGHT}  (missed seizure = 5x penalty)")
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 1b: DVC Pull (Optional)

If your dataset is versioned with DVC, this pulls the latest version from Google Drive. Skips download if already up to date.

In [ ]:
# OPTIONAL: Pull latest dataset from DVC (Google Drive remote)
# Uncomment below if using DVC instead of Kaggle Dataset:

# !pip install -q dvc[gdrive]
# !git clone https://github.com/pinkprincess536/eeg.git /kaggle/working/repo
# %cd /kaggle/working/repo
# !dvc remote add -d myremote gdrive://MyDrive/EEG_PROJECT/.dvc/cache
# !dvc pull
# DATA_DIR = "/kaggle/working/repo/processed"

print("Using Kaggle Dataset (not DVC pull).")

## Cell 2: Load Preprocessed Data

Loads .npy files saved by the Colab preprocessing notebook.
No EDF files. No MNE. No filtering. Just numpy arrays.

In [ ]:
import os

print("Loading preprocessed data...")

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print(f"\nTrain: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape},  y={y_test.shape}")

# Print class distribution
ut, ct = np.unique(y_train, return_counts=True)
print(f"Train classes: {dict(zip(ut, ct))}")
ut, ct = np.unique(y_test, return_counts=True)
print(f"Test classes:  {dict(zip(ut, ct))}")

# Print metadata if available
info_path = os.path.join(DATA_DIR, "info.txt")
if os.path.exists(info_path):
    with open(info_path, "r") as f:
        print(f"\n--- Metadata ---\n{f.read()}")
else:
    print("\nNo info.txt found (optional metadata file)")

## Cell 3: Data Augmentation

Conservative augmentation for seizure windows only.
Time shift +-200ms, amplitude x0.85-1.15, Gaussian noise sigma=0.01.
Applied on-the-fly during training, not to non-seizure windows.

In [ ]:
def augment_seizure(windows, seed=None):
    """Augment seizure windows: time shift + scale + noise."""
    if seed is not None:
        np.random.seed(seed)
    aug = windows.copy()
    B, C, T = aug.shape
    for b in range(B):
        shift = np.random.randint(-51, 51)
        aug[b] = np.roll(aug[b], shift, axis=-1)
        aug[b] *= np.random.uniform(0.85, 1.15)
        aug[b] += np.random.randn(C, T).astype(np.float32) * 0.01
    return aug

print("Augmentation function ready.")

## Cell 4: PyTorch Tensors + DataLoader

1D CNN input shape: (batch, 23 channels, 1792 time samples).

In [ ]:
Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test,  dtype=torch.float32)
yte = torch.tensor(y_test,  dtype=torch.long)

train_ds = TensorDataset(Xtr, ytr)
test_ds  = TensorDataset(Xte, yte)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Input shape: {Xtr.shape}  (samples, 23 channels, 1792 time)")
print(f"Train batches: {len(train_loader)} | Test: {len(test_loader)}")

## Cell 5: 1D CNN Model

3 Conv1d blocks (k7, k5, k3) + BatchNorm + Dropout + AdaptiveAvgPool1d(16)

```
Block 1: Conv1d(23->32,k=7) -> BN -> ReLU -> MaxPool(4)  (1792->448)
Block 2: Conv1d(32->64,k=5) -> BN -> ReLU -> MaxPool(4)  (448->112)
Block 3: Conv1d(64->128,k=3)-> BN -> ReLU -> AdaptiveAvgPool1d(16)
Flatten -> Dropout(0.4) -> FC(2048->64) -> Dropout(0.3) -> FC(64->2)
```

In [ ]:
n_ch = X_train.shape[1]  # auto-detect channel count from data

class EEGCNN1D(nn.Module):
    def __init__(self, n_channels=n_ch, n_classes=2):
        super().__init__()
        # Block 1
        self.conv1 = nn.Conv1d(n_channels, 32, kernel_size=7, padding=3)
        self.bn1   = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(4)

        # Block 2
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2   = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(4)

        # Block 3
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm1d(128)
        self.adapt = nn.AdaptiveAvgPool1d(16)

        # Classifier
        self.drop1 = nn.Dropout(0.4)
        self.fc1   = nn.Linear(128 * 16, 64)
        self.drop2 = nn.Dropout(0.3)
        self.fc2   = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))  # (B,32,448)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # (B,64,112)
        x = F.relu(self.bn3(self.conv3(x)))               # (B,128,112)
        x = self.adapt(x)                                 # (B,128,16)
        x = torch.flatten(x, 1)                           # (B,2048)
        x = self.drop1(x)
        x = F.relu(self.fc1(x))                           # (B,64)
        x = self.drop2(x)
        x = self.fc2(x)                                   # (B,2)
        return x

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EEGCNN1D().to(device)

# Verify
dummy = torch.randn(1, n_ch, X_train.shape[2]).to(device)
with torch.no_grad():
    out = model(dummy)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} params | Device: {device}")
print(f"Input {tuple(dummy.shape)} -> Output {tuple(out.shape)}  OK!")

## Cell 6: Train 1D CNN with Augmentation

Augmentation applied ONLY to seizure windows.
1 conservative copy per seizure window per epoch.

In [ ]:
# Weighted loss: missing a seizure costs 5x more than a false alarm
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, SEIZURE_WEIGHT]).to(device))
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Track metrics per epoch (on training set -- no test leakage)
train_recall = []
train_precision = []
train_specificity = []
train_accuracy = []
train_epoch = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    n_batches = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        # Augment seizure windows only
        sm = (y == 1)
        if sm.any():
            si = x[sm].cpu().numpy()
            sl = y[sm]
            aug_np = augment_seizure(si)
            aug_t = torch.tensor(aug_np, dtype=torch.float32).to(device)
            x_aug = torch.cat([x, aug_t], dim=0)
            y_aug = torch.cat([y, sl], dim=0)
        else:
            x_aug, y_aug = x, y

        optimizer.zero_grad()
        outputs = model(x_aug)
        loss = criterion(outputs, y_aug)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1

    avg_loss = running_loss / n_batches

    # Evaluate on TRAINING set (no test leakage, just monitoring) ---
    model.eval()
    all_preds_tr, all_true_tr = [], []
    with torch.no_grad():
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            o = model(x)
            _, p = torch.max(o, 1)
            all_preds_tr.extend(p.cpu().numpy())
            all_true_tr.extend(y.cpu().numpy())

    all_preds_tr = np.array(all_preds_tr)
    all_true_tr  = np.array(all_true_tr)

    # Recall = TP / (TP + FN)
    sez_mask = all_true_tr == 1
    tp = (all_preds_tr[sez_mask] == 1).sum()
    fn = (all_preds_tr[sez_mask] == 0).sum()
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0

    # Precision = TP / (TP + FP)
    pred_sez = all_preds_tr == 1
    fp = (all_true_tr[pred_sez] == 0).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0

    # Specificity = TN / (TN + FP)
    non_mask = all_true_tr == 0
    tn = (all_preds_tr[non_mask] == 0).sum()
    fp2 = (all_preds_tr[non_mask] == 1).sum()
    spec = tn / (tn + fp2) if (tn + fp2) > 0 else 0

    # Accuracy
    acc = (all_preds_tr == all_true_tr).mean()

    train_recall.append(rec)
    train_precision.append(prec)
    train_specificity.append(spec)
    train_accuracy.append(acc)
    train_epoch.append(epoch + 1)

    model.train()  # back to train mode
    print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | "
          f"Recall: {rec:.3f} | Precision: {prec:.3f} | "
          f"Specificity: {spec:.3f} | Acc: {acc:.3f}")

print("\nTraining complete.")

## Cell 9: Evaluate on Unseen Patients

Metrics: test set only — no leakage.

In [ ]:
model.eval()
correct, total = 0, 0
all_preds, all_true = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_true.extend(y.cpu().numpy())
        total += y.size(0)
        correct += (predicted == y).sum().item()

all_preds = np.array(all_preds)
all_true  = np.array(all_true)
accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}% ({correct}/{total})")

# Sensitivity (recall)
seizure_mask = all_true == 1
if seizure_mask.sum() > 0:
    tp = (all_preds[seizure_mask] == 1).sum()
    fn = (all_preds[seizure_mask] == 0).sum()
    sens = tp / (tp + fn)
    print(f"Sensitivity (recall): {sens:.2f}  (caught {tp}/{tp+fn} seizures)")

# Specificity
non_mask = all_true == 0
if non_mask.sum() > 0:
    tn = (all_preds[non_mask] == 0).sum()
    fp = (all_preds[non_mask] == 1).sum()
    spec = tn / (tn + fp)
    print(f"Specificity:         {spec:.2f}  (rejected {tn}/{tn+fp} non-seizures)")

## Cell 8: Training Summary

Epoch-by-epoch comparison. Recall = % of seizures caught. Precision = % of seizure predictions that were correct. Specificity = % of non-seizures correctly ignored.

In [ ]:
print(f"{'Epoch':<6} {'Recall':<8} {'Precision':<10} {'Specificity':<12} {'Accuracy':<9}")
print("-" * 50)
for i in range(len(train_epoch)):
    print(f"{train_epoch[i]:<6}"
          f"{train_recall[i]:<8.3f}"
          f"{train_precision[i]:<10.3f}"
          f"{train_specificity[i]:<12.3f}"
          f"{train_accuracy[i]:<9.3f}")